## Preprocessing

Cleans the six datasets and saves them to `data/processed/`. Scaling and SMOTE are left to the model pipeline in notebook 05, so that they are fitted on training data only.

In [1]:
import io
import re
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

QUESTIONS = [f"A{i}" for i in range(1, 11)]
FEATURES = QUESTIONS + ["age", "male", "jaundice", "family_asd"]

In [2]:
def read_arff(path):
    text = Path(path).read_text(encoding="latin-1")
    header, body = re.split("@data", text, flags=re.IGNORECASE)
    cols = [line.split()[1] for line in header.splitlines()
            if line.lower().startswith("@attribute")]
    return pd.read_csv(io.StringIO(body), header=None, names=cols,
                       quotechar="'", na_values="?", skipinitialspace=True)


raw = {
    "toddler": pd.read_csv(RAW / "toddler/Toddler Autism dataset July 2018.csv"),
    "child": read_arff(RAW / "child/Autism-Child-Data.arff"),
    "adolescent": read_arff(RAW / "adolescent/Autism-Adolescent-Data.arff"),
    "adult": read_arff(RAW / "adult/Autism-Adult-Data.arff"),
    "toddler_saudi": pd.read_csv(RAW / "toddler_saudi/Autism Spectrum Disorder Screening Data for Toddlers in Saudi Arabia Data Set.csv"),
    "adult_kaggle": pd.read_csv(RAW / "adult_kaggle/train.csv", na_values="?"),
}

### 1. One schema for all datasets

The toddler files use their own column names; the other four follow the UCI layout. Yes/no fields become 0/1 and gender becomes `male` (1 = male). The Saudi data has no jaundice or ethnicity columns, so those are left empty.

In [3]:
RENAME = {
    "toddler": {"Age_Mons": "age", "Sex": "gender", "Ethnicity": "ethnicity",
                "Jaundice": "jaundice", "Family_mem_with_ASD": "family_asd",
                "Qchat-10-Score": "score", "Class/ASD Traits ": "asd"},
    "toddler_saudi": {"Age": "age", "Gender": "gender",
                      "Family member with ASD history": "family_asd",
                      "Screening Score": "score", "Class": "asd"},
    "uci": {**{f"A{i}_Score": f"A{i}" for i in range(1, 11)},
            "jundice": "jaundice", "austim": "family_asd", "result": "score", "Class/ASD": "asd"},
}


def yes_no(s):
    return s.astype(str).str.strip().str.lower().isin(["yes", "1"]).astype(int)


def harmonise(name, df):
    df = df.rename(columns=RENAME.get(name, RENAME["uci"]))
    out = df[QUESTIONS].astype(int)
    out["age"] = pd.to_numeric(df["age"], errors="coerce")
    out["male"] = df["gender"].str.strip().str.lower().str.startswith("m").astype(int)
    out["jaundice"] = yes_no(df["jaundice"]) if "jaundice" in df else pd.NA
    out["family_asd"] = yes_no(df["family_asd"])
    out["ethnicity"] = df["ethnicity"] if "ethnicity" in df else pd.NA
    out["score"] = df["score"]  # kept for analysis only, never a feature
    out["asd"] = yes_no(df["asd"])
    return out


data = {name: harmonise(name, df) for name, df in raw.items()}
rows_before = {name: len(df) for name, df in data.items()}
data["adult"].head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,age,male,jaundice,family_asd,ethnicity,score,asd
0,1,1,1,1,0,0,1,1,0,0,26.0,0,0,0,White-European,6,0
1,1,1,0,1,0,0,0,1,0,1,24.0,1,0,1,Latino,5,0
2,1,1,0,1,1,0,1,1,1,1,27.0,1,1,1,Latino,8,1
3,1,1,0,1,0,0,1,1,0,1,35.0,0,0,1,White-European,6,0
4,1,0,0,0,0,0,0,1,0,0,40.0,0,0,0,NaN,2,0


### 2. Duplicates

After dropping the ID column, some records are exact copies. Two people can give the same answers, but with only 100–1,000 rows a copied record falling in both the training and test fold makes results look better than they are, so exact duplicates are removed.

In [4]:
for name, df in data.items():
    n = df.duplicated().sum()
    data[name] = df.drop_duplicates().reset_index(drop=True)
    print(f"{name:<14} {n:3d} duplicates removed")

toddler         81 duplicates removed
child            8 duplicates removed
adolescent       3 duplicates removed
adult           10 duplicates removed
toddler_saudi   63 duplicates removed
adult_kaggle     0 duplicates removed


### 3. Age

A handful of ages are missing, and the adult file contains an age of 383. The Kaggle adult data, despite being labelled "18 and more", contains ages from 2 to 89 because it was synthetically generated. Rows outside the age range of each questionnaire are dropped.

In [5]:
AGE_RANGE = {"toddler": (12, 36), "child": (4, 11), "adolescent": (12, 16), "adult": (17, 100)}

for name, df in data.items():
    low, high = AGE_RANGE[name.split("_")[0]]
    keep = df["age"].between(low, high)
    data[name] = df[keep].reset_index(drop=True)
    print(f"{name:<14} {(~keep).sum():3d} rows dropped (age missing or outside {low}-{high})")

toddler          0 rows dropped (age missing or outside 12-36)
child            4 rows dropped (age missing or outside 4-11)
adolescent       0 rows dropped (age missing or outside 12-16)
adult            3 rows dropped (age missing or outside 17-100)
toddler_saudi    0 rows dropped (age missing or outside 12-36)
adult_kaggle   194 rows dropped (age missing or outside 17-100)


### 4. Ethnicity

Ethnicity is **not** given to the models. It is only used later to check whether the models make more mistakes for some groups. The labels are spelled inconsistently (`White-European` vs `White European`, `others` vs `Others`), so they are tidied up; missing values become `Unknown`.

In [6]:
def clean_ethnicity(s):
    s = s.astype("string").str.strip().str.replace("-", " ").str.title()
    return s.replace({"Others": "Other", "Pacifica": "Pasifika"}).fillna("Unknown")


for df in data.values():
    df["ethnicity"] = clean_ethnicity(df["ethnicity"])

data["adult"]["ethnicity"].value_counts()

ethnicity
White European    227
Asian             123
Unknown            91
Middle Eastern     90
Black              43
South Asian        36
Other              31
Latino             20
Hispanic           13
Pasifika           11
Turkish             6
Name: count, dtype: Int64

### 5. Model inputs

| Column | Decision | Why |
|---|---|---|
| A1–A10 | keep | the screening answers |
| age, male, jaundice, family_asd | keep | basic details collected by the app |
| score / result | **drop** | the label is computed from it (see notebook 02) |
| ethnicity | drop | bias check only; also many missing values |
| country, relation, used_app_before, age_desc | drop | not about the person, or constant |

Every input is now 0/1 except `age`, so one-hot encoding is not needed.

In [7]:
print("features:", FEATURES)

features: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'age', 'male', 'jaundice', 'family_asd']


### 6. Result and save

In [8]:
summary = pd.DataFrame({
    name: {"rows before": rows_before[name], "rows after": len(df),
           "ASD traits %": round(100 * df["asd"].mean(), 1)}
    for name, df in data.items()
}).T
summary.astype({"rows before": int, "rows after": int})

,rows before,rows after,ASD traits %
toddler,1054,973,70.9
child,292,280,47.5
adolescent,104,101,59.4
adult,704,691,26.8
toddler_saudi,506,443,69.1
adult_kaggle,800,606,21.3


In [9]:
PROCESSED.mkdir(parents=True, exist_ok=True)
for name, df in data.items():
    df.to_csv(PROCESSED / f"{name}.csv", index=False)

sorted(p.name for p in PROCESSED.glob("*.csv"))

['adolescent.csv',
 'adult.csv',
 'adult_kaggle.csv',
 'child.csv',
 'toddler.csv',
 'toddler_saudi.csv']

The class balance differs a lot between groups: about 70 % positive for toddlers, close to 50/50 for children, and under 30 % for adults. This is handled with SMOTE inside the training folds (notebook 05).